In [54]:
%load_ext autoreload
%autoreload 2
# %matplotlib inline

import os
while 'notebooks' in os.getcwd():
    os.chdir("../")

import torch
from torch import nn, einsum

import quantus
import gc
import torch.nn.functional as F
import pandas as pd

from lib.helpers import plot_example_grid, short_titles_map
from lib.attributions import GradientAscentDiff, PullbackAscentDiff, \
    quantus_pullback_ascent_diff_explain_func
from lib.setup import setup_notebook
from lib.defaults import get_default_kwargs
from lib.surrogates import LayerNorm2d, PVTAttention, soften_module_inplace_
from lib.evaluator import QuantusEvaluator, default_explainers, default_metrics

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
short_metrics_map = {
    "infidelity": "Infidelity",
    "faithfulness_correlation": "Faith.Corr",
    "monotonicity_correlation": "Mono.Corr",
    "max_sensitivity": "Max.Sens",
    "random_logit": "Rand.Logit",
}
# explainers = ["SoftPullback", "PullbackAscent", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]
explainers = ["Gradient", "SoftPullback", "GradientAscent", "PullbackAscent", "SmoothGrad", "SmoothPullback", "FusionGrad", "FusionPullback", "IntegratedGradients", "GradientShap", "DeepLift"]

In [56]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# NumPy 2 pickles refer to numpy._core. Expose equivalent modules when this
# notebook runs with NumPy 1 so that saved results remain readable.
if int(np.__version__.split(".", maxsplit=1)[0]) < 2:
    import numpy.core as _numpy_core
    import numpy.core.multiarray as _numpy_multiarray
    import numpy.core.numeric as _numpy_numeric

    sys.modules.setdefault("numpy._core", _numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", _numpy_multiarray)
    sys.modules.setdefault("numpy._core.numeric", _numpy_numeric)

def _as_float_array(value):
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        arr = np.asarray(value, dtype=float).reshape(-1)
    else:
        arr = np.asarray([value], dtype=float)
    return arr[np.isfinite(arr)]

def _format_val(x, precision=3):
    if np.isnan(x):
        return ""
    if abs(x) >= 1e4 or (abs(x) > 0 and abs(x) < 1e-3):
        return f"{x:.2e}"
    s = f"{x:.{precision}f}"
    return s.rstrip("0").rstrip(".") if "." in s else s

def _format_mean_std(arr, precision=3):
    if arr.size == 0:
        return ""
    mean = arr.mean()
    std = arr.std()
    mean_str = _format_val(mean, precision=precision)
    std_str = _format_val(std, precision=max(1, precision - 1))
    return f"{mean_str}±{std_str}"

def summarize_ablations_pickle(pkl_path, show=True):
    pkl_path = Path(pkl_path)
    if not pkl_path.exists():
        raise FileNotFoundError(f"File not found: {pkl_path}")

    df = pd.read_pickle(pkl_path)
    metric_cols = [col for col in short_metrics_map if col in df.columns]
    group_cols = ["model_name", "parameter", "value", "default_value", "explainer"]

    summary_rows = []
    for _, row in df.iterrows():
        base = {col: row[col] for col in group_cols}
        for metric in metric_cols:
            arr = _as_float_array(row[metric])
            base[short_metrics_map[metric]] = _format_mean_std(arr, precision=3)
        summary_rows.append(base)

    summary = (
        pd.DataFrame(summary_rows)
        .sort_values(["model_name", "parameter", "value", "explainer"])
        .reset_index(drop=True)
    )

    csv_path = pkl_path.with_name(pkl_path.stem + "_summary.csv")
    summary.to_csv(csv_path, index=False)

    print(f"Loaded: {pkl_path}")
    print(f"Saved CSV: {csv_path}")
    if show:
        display(summary)
    return summary, csv_path

In [57]:
def merge_with_priority(df_big, df_small, key_col="Faith.Corr"):
    # we merge results as we re-genenerated the faithfulness_correlation metric for different set of patches
    # the previous selection had too large patches, which inflated the correlation score
    result = df_big.loc[df_small.index].copy()
    result[key_col] = df_small[key_col]
    return result

def process_df(
    df_path,
    precision=3,
    uncertainty="std",
    confidence_level=0.95,
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    """Use uncertainty='bootstrap_ci' to report mean [95% CI] instead of mean±std."""
    selected_columns = list(short_metrics_map.keys())
    
    df = QuantusEvaluator.load_results(df_path)
    df = df[
        [col for col in selected_columns if col in df.columns]
    ].rename(columns=short_metrics_map)
    df = df.loc[[idx for idx in explainers if idx in df.index]]
    df = df.rename(index=short_titles_map)
    
    print("num_samples:", len(df.iloc[0].iloc[0]))
    
    df = QuantusEvaluator.summarize_results(
        df,
        precision=precision,
        uncertainty=uncertainty,
        confidence_level=confidence_level,
        bootstrap_resamples=bootstrap_resamples,
        bootstrap_seed=bootstrap_seed,
    )
    return df

def add_focus_to_df(df, focus_pkl_path):
    focus_df = pd.read_pickle(focus_pkl_path)

    focus_summary = focus_df.groupby("explainer")["score"].apply(
        lambda scores: (
            f"{scores.mean():.3f}±{scores.std():.2f}"
            if len(scores) > 1 else f"{scores.mean():.2f}±nan"
        )
    )
    focus_summary.index = focus_summary.index.map(short_titles_map)

    result = df.drop(columns="Focus", errors="ignore").copy()
    result.insert(1, "Focus", focus_summary.reindex(result.index))
    return result

In [58]:
vgg_df = process_df("results/quantus_vgg_fc_nips_20_50_vgg11_bn")
vgg_df = add_focus_to_df(
    vgg_df,
    "results/focus/quantus_vgg_focus_1000_model_name=vgg11_bn.pkl",
)
vgg_df.to_csv('results/vgg_df_20_50.csv', index=True)
vgg_df

num_samples: 1000


,Infidelity,Focus,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
Grad,100.090±107.94,0.544±0.13,0.457±0.35,0.316±0.37,0.898±0.14,-0.053±0.29
SoftPb,16.303±31.68,0.614±0.11,0.606±0.30,0.375±0.39,0.212±0.08,-0.064±0.33
GA,18.523±35.24,0.557±0.10,0.508±0.30,0.234±0.32,1.217±0.06,0.001±0.07
PA,5.616±13.18,0.623±0.11,0.570±0.29,0.312±0.37,0.390±0.11,0.136±0.13
SmoothG,52.044±70.80,0.562±0.11,0.513±0.34,0.385±0.36,0.781±0.11,-0.025±0.19
SmoothPb,14.030±30.57,0.625±0.12,0.545±0.32,0.309±0.40,0.398±0.11,-0.048±0.32
FusionG,38.651±60.23,0.561±0.11,0.506±0.34,0.363±0.34,0.955±0.13,-0.013±0.16
FusionPb,16.056±34.46,0.614±0.12,0.513±0.33,0.315±0.39,0.513±0.11,-0.035±0.29
IG,75.796±92.29,0.547±0.16,0.434±0.36,0.379±0.41,0.752±0.15,-0.046±0.27
GradShap,76.050±94.76,0.545±0.16,0.430±0.36,0.381±0.41,1.086±0.22,-0.042±0.26


In [59]:
pvt_df_fc = process_df("results/quantus_pvt_fc_nips_20_50_pvt_v2_b1")
pvt_df = process_df("results/quantus_pvt_it_nips_20_50_pvt_v2_b1")
pvt_merged_df = merge_with_priority(pvt_df, pvt_df_fc)
pvt_merged_df = add_focus_to_df(
    pvt_merged_df,
    "results/focus/quantus_pvt_focus_model_name=pvt_v2_b1.pkl",
)
pvt_merged_df.to_csv('results/pvt_df_20_50.csv', index=True)
pvt_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Focus,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
Grad,8.914±7.89,0.557±0.18,0.105±0.33,0.185±0.43,1.034±0.16,-0.030±0.30
SoftPb,6.264±6.40,0.617±0.18,0.119±0.34,0.164±0.43,1.066±0.18,-0.006±0.35
GA,4.506±4.07,0.551±0.10,0.118±0.33,0.164±0.35,1.242±0.07,0.006±0.06
PA,1.634±1.03,0.554±0.10,0.122±0.36,0.219±0.33,0.855±0.12,0.121±0.11
SmoothG,8.798±7.08,0.594±0.11,0.096±0.38,0.280±0.38,0.569±0.10,-0.012±0.13
SmoothPb,4.974±5.46,0.635±0.10,0.102±0.39,0.231±0.38,0.519±0.09,0.008±0.23
FusionG,6.673±5.60,0.606±0.10,0.101±0.39,0.262±0.38,0.979±0.16,0.001±0.11
FusionPb,5.156±5.72,0.614±0.11,0.093±0.39,0.200±0.38,0.753±0.11,0.007±0.19
IG,12.578±8.39,0.571±0.19,0.068±0.32,0.170±0.45,1.374±0.37,0.008±0.33
GradShap,12.433±8.45,0.562±0.19,0.067±0.32,0.169±0.45,2.471±1.56,0.003±0.34


In [60]:
resnet_df_fc = process_df("results/quantus_resnet_fc_nips_20_50_resnet50")
resnet_df = process_df("results/quantus_resnet_it_nips_20_50_resnet50")
resnet_merged_df = merge_with_priority(resnet_df, resnet_df_fc)
resnet_merged_df = add_focus_to_df(
    resnet_merged_df,
    "results/focus/quantus_resnet_focus_1000_model_name=resnet50.pkl",
)
resnet_merged_df.to_csv('results/resnet_df_20_50.csv', index=True)
resnet_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Focus,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
Grad,83.294±68.15,0.634±0.14,0.241±0.36,0.275±0.39,0.952±0.17,-0.032±0.23
SoftPb,5.989±5.67,0.746±0.13,0.389±0.37,0.420±0.39,0.119±0.04,-0.062±0.42
GA,29.480±35.36,0.584±0.10,0.278±0.37,0.231±0.32,1.267±0.04,0.002±0.05
PA,5.384±4.83,0.726±0.13,0.382±0.37,0.421±0.40,0.244±0.09,0.212±0.19
SmoothG,66.883±58.16,0.640±0.12,0.331±0.40,0.364±0.36,0.639±0.08,-0.024±0.20
SmoothPb,8.122±10.13,0.713±0.16,0.326±0.39,0.352±0.40,0.220±0.05,-0.065±0.40
FusionG,58.514±57.80,0.634±0.12,0.344±0.39,0.357±0.35,0.855±0.09,-0.014±0.17
FusionPb,10.131±14.55,0.691±0.16,0.306±0.40,0.350±0.40,0.362±0.08,-0.051±0.39
IG,66.830±61.21,0.626±0.17,0.244±0.38,0.350±0.41,0.788±0.21,-0.029±0.23
GradShap,69.072±62.93,0.619±0.17,0.233±0.38,0.344±0.41,1.400±0.41,-0.023±0.24


# Ablations

In [64]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    # "ablations 2/quantus_pvt_20_50_ablations_model_name=pvt_v2_b1_n_batches=50_test_mode=full.pkl"
    "results/ablations/quantus_pvt_20_50_ablations.pkl"
)

Loaded: results\ablations\quantus_pvt_20_50_ablations.pkl
Saved CSV: results\ablations\quantus_pvt_20_50_ablations_summary.csv


,model_name,parameter,value,default_value,explainer,Infidelity,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
0,pvt_v2_b1,K,1,5,PullbackAscent,6.224±6.35,0.119±0.34,0.164±0.43,1.11±0.16,-0.006±0.35
1,pvt_v2_b1,K,2,5,PullbackAscent,2.686±2.78,0.134±0.35,0.215±0.38,1.008±0.13,0.044±0.2
2,pvt_v2_b1,K,3,5,PullbackAscent,1.926±1.64,0.134±0.36,0.224±0.35,0.939±0.12,0.073±0.15
3,pvt_v2_b1,K,10,5,PullbackAscent,1.556±0.96,0.103±0.36,0.221±0.33,0.8±0.12,0.139±0.07
4,pvt_v2_b1,alpha,5,20,PullbackAscent,1.962±1.66,0.133±0.36,0.192±0.38,0.85±0.13,0.111±0.13
5,pvt_v2_b1,alpha,10,20,PullbackAscent,1.702±1.11,0.13±0.36,0.209±0.35,0.849±0.13,0.114±0.11
6,pvt_v2_b1,alpha,40,20,PullbackAscent,1.616±1.04,0.11±0.36,0.227±0.33,0.867±0.11,0.105±0.1
7,pvt_v2_b1,default,default,default,PullbackAscent,1.619±1.03,0.122±0.36,0.229±0.33,0.855±0.12,0.113±0.11
8,pvt_v2_b1,default,default,default,SoftPullback,6.183±6.12,0.119±0.34,0.164±0.43,1.065±0.18,-0.006±0.35
9,pvt_v2_b1,tau_attention,0.5,1.0,PullbackAscent,1.681±1.1,0.12±0.36,0.22±0.35,0.927±0.11,0.101±0.1


In [63]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    # "ablations 2/quantus_vgg_20_50_ablations_model_name=vgg11_bn_n_batches=50_test_mode=full.pkl"
    "results/ablations/quantus_resnet_20_50_ablations.pkl"
)

Loaded: results\ablations\quantus_resnet_20_50_ablations.pkl
Saved CSV: results\ablations\quantus_resnet_20_50_ablations_summary.csv


,model_name,parameter,value,default_value,explainer,Infidelity,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
0,resnet50,K,1,5,PullbackAscent,5.955±5.63,0.389±0.37,0.418±0.39,0.115±0.03,-0.006±0.43
1,resnet50,K,2,5,PullbackAscent,5.471±5.01,0.396±0.37,0.411±0.39,0.139±0.05,0.152±0.31
2,resnet50,K,3,5,PullbackAscent,5.352±4.86,0.393±0.37,0.418±0.4,0.179±0.07,0.199±0.26
3,resnet50,K,10,5,PullbackAscent,5.527±5.05,0.358±0.36,0.423±0.4,0.358±0.11,0.219±0.14
4,resnet50,alpha,5,20,PullbackAscent,5.393±4.94,0.403±0.37,0.418±0.4,0.179±0.08,0.261±0.25
5,resnet50,alpha,10,20,PullbackAscent,5.327±4.85,0.396±0.37,0.42±0.4,0.208±0.09,0.261±0.22
6,resnet50,alpha,40,20,PullbackAscent,5.472±5.08,0.362±0.37,0.415±0.4,0.305±0.1,0.173±0.19
7,resnet50,default,default,default,PullbackAscent,5.349±4.85,0.382±0.37,0.428±0.4,0.245±0.09,0.228±0.2
8,resnet50,default,default,default,SoftPullback,5.974±5.7,0.389±0.37,0.419±0.39,0.118±0.04,-0.006±0.43
9,resnet50,tau_maxpool,0.01,0.3,PullbackAscent,5.338±4.86,0.368±0.37,0.361±0.39,0.489±0.13,0.2±0.19


In [65]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    # "ablations 2/quantus_vgg_20_50_ablations_model_name=vgg11_bn_n_batches=50_test_mode=full.pkl"
    "results/ablations/quantus_vgg_20_50_ablations.pkl"
)

Loaded: results\ablations\quantus_vgg_20_50_ablations.pkl
Saved CSV: results\ablations\quantus_vgg_20_50_ablations_summary.csv


,model_name,parameter,value,default_value,explainer,Infidelity,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
0,vgg11_bn,K,1,5,PullbackAscent,15.933±29.73,0.606±0.3,0.366±0.39,0.208±0.07,-0.005±0.32
1,vgg11_bn,K,2,5,PullbackAscent,8.998±20.86,0.601±0.29,0.329±0.38,0.251±0.08,0.101±0.21
2,vgg11_bn,K,3,5,PullbackAscent,6.798±17.18,0.594±0.29,0.32±0.38,0.304±0.1,0.136±0.18
3,vgg11_bn,K,10,5,PullbackAscent,5.09±8.63,0.537±0.3,0.315±0.36,0.521±0.13,0.165±0.1
4,vgg11_bn,alpha,5,20,PullbackAscent,6.321±13.88,0.588±0.29,0.321±0.39,0.249±0.09,0.211±0.17
5,vgg11_bn,alpha,10,20,PullbackAscent,5.738±13.28,0.577±0.29,0.314±0.38,0.299±0.09,0.206±0.15
6,vgg11_bn,alpha,40,20,PullbackAscent,6.15±15.74,0.57±0.29,0.319±0.36,0.538±0.12,0.112±0.13
7,vgg11_bn,default,default,default,PullbackAscent,5.624±13.24,0.57±0.29,0.329±0.37,0.389±0.11,0.166±0.14
8,vgg11_bn,default,default,default,SoftPullback,15.702±29.89,0.606±0.3,0.369±0.39,0.213±0.08,-0.005±0.32
9,vgg11_bn,tau_maxpool,0.01,0.3,PullbackAscent,4.097±4.51,0.53±0.29,0.289±0.33,0.765±0.13,0.133±0.13
